# VisDrone learning-rate search

This shared notebook runs the same LR-only protocol for one supported primary
model. Search manifests are drawn exclusively from official train. Search
checkpoints are isolated and are never registered as final benchmark runs.


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

SMOKE_TEST = os.environ.get("SMOKE_TEST", "0").lower() in {"1", "true", "yes", "on"}
try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False
REPOSITORY_URL = os.environ.get(
    "BENCHMARK_REPOSITORY_URL",
    "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git",
)
REPOSITORY_BRANCH = os.environ.get("BENCHMARK_REPOSITORY_BRANCH", "main")
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/aerial-object-detection-benchmark")
    if not (REPO_DIR / ".git").is_dir():
        subprocess.run(
            ["git", "clone", "--branch", REPOSITORY_BRANCH, REPOSITORY_URL, str(REPO_DIR)],
            check=True,
        )
else:
    REPO_DIR = Path(os.environ.get("BENCHMARK_REPO_ROOT", Path.cwd())).resolve()
if not (REPO_DIR / "pyproject.toml").is_file():
    raise RuntimeError(f"Repository root is invalid: {REPO_DIR}")
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if IS_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-dataset-colab.txt"],
        check=True,
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
DRIVE_ROOT = os.environ.get(
    "VISDRONE_DRIVE_ROOT",
    "/content/drive/MyDrive/visdrone_architecture_benchmark"
    if IS_COLAB
    else str(REPO_DIR / ".notebook-smoke"),
)
from src.paths import ProjectPaths
from src.reproducibility import seed_everything
from src.utils.environment import collect_environment
paths = ProjectPaths.from_value(DRIVE_ROOT).create()
seed_everything(42)
print({"repo": str(REPO_DIR), "storage": str(paths.root), "smoke_test": SMOKE_TEST})
collect_environment()


## Configuration

Change `MODEL_ID` for the model-day being run.


In [ ]:
MODEL_ID = "rtdetrv2_l"
DATASET_TRACK = "2class"

RUN_LR_RANGE_TEST = True
RUN_BOUNDARY_EXTENSION = False

SEARCH_SEED = 42
IMAGE_SIZE = 640
EFFECTIVE_BATCH_SIZE = 8
SEARCH_MAX_EPOCHS = 15

LR_CANDIDATES = 9
PROMOTION_RUNGS = [
    {"epoch": 2, "keep": 5},
    {"epoch": 5, "keep": 3},
    {"epoch": 10, "keep": 2},
    {"epoch": 15, "keep": 1},
]

ONE_DAY_PER_MODEL = True
ALLOW_OVER_BUDGET_RUN = False
PER_DEVICE_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
START_EXPENSIVE_STAGE = False
if SMOKE_TEST:
    START_EXPENSIVE_STAGE = False
assert DATASET_TRACK == "2class"
assert EFFECTIVE_BATCH_SIZE == PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS


## Validate manifests, model identity, and baseline optimizer


In [ ]:
from src.models.registry import create_adapter, load_model_config
from src.training.lr_search import (
    SUPPORTED_PRIMARY_MODELS,
    generate_lr_candidates,
)
from src.training.lr_workflow import LRControlledBenchmark

assert MODEL_ID in SUPPORTED_PRIMARY_MODELS
workflow = LRControlledBenchmark(REPO_DIR, DRIVE_ROOT)
split_summary = workflow.prepare_manifests()
model_config = load_model_config(MODEL_ID, REPO_DIR)
adapter = create_adapter(MODEL_ID, "cpu")
baseline = workflow.resolve_baseline(MODEL_ID)
default_candidates = generate_lr_candidates(baseline.learning_rate)
print("Framework:", model_config["framework"])
print("Adapter:", type(adapter).__name__)
print("Baseline audit:", baseline)
print("Search split checks:", split_summary["verification"])
print("Default LR candidates:", default_candidates)


## Workload contract


In [ ]:
SEARCH_EPOCH_EQUIVALENTS = 9 * 2 + 5 * 3 + 3 * 5 + 2 * 5
print("Search train epoch-equivalents:", SEARCH_EPOCH_EQUIVALENTS)
print("Search validation passes:", SEARCH_EPOCH_EQUIVALENTS)
calibration_path = paths.lr_search_checkpoints / MODEL_ID / "calibration.json"
if calibration_path.exists():
    import json
    calibration = json.loads(calibration_path.read_text())
    estimate = workflow.workload_estimate(
        calibration,
        range_optimizer_steps=300 if RUN_LR_RANGE_TEST else 0,
        batch_size=PER_DEVICE_BATCH_SIZE,
        accumulation=GRADIENT_ACCUMULATION_STEPS,
    )
    print("Measured workload estimate:", estimate)
    if ONE_DAY_PER_MODEL and estimate["total_hours"] > 24:
        print("WARNING: estimated protocol exceeds 24 hours.")
        print("Set ALLOW_OVER_BUDGET_RUN=True to opt in explicitly.")
else:
    print("A one-epoch calibration will run before the search starts.")


## Run or resume successive halving


In [ ]:
if START_EXPENSIVE_STAGE:
    from src.notebook_utils import require_gpu, require_model_environment
    require_model_environment(
        "rtdetr" if MODEL_ID == "rtdetrv2_l" else "openmmlab"
    )
    require_gpu(MODEL_ID)
    search_result = workflow.run_search(
        MODEL_ID,
        batch_size=PER_DEVICE_BATCH_SIZE,
        accumulation=GRADIENT_ACCUMULATION_STEPS,
        run_lr_range_test=RUN_LR_RANGE_TEST,
        run_boundary_extension=RUN_BOUNDARY_EXTENSION,
        allow_over_budget_run=ALLOW_OVER_BUDGET_RUN,
    )
    print("Promotion decisions:")
    for decision in search_result["state"]["rung_decisions"]:
        print(decision)
    print("Selected:", search_result["selected"])
else:
    print("Expensive stage is OFF. Validation and candidate preview completed.")
    print("Set START_EXPENSIVE_STAGE=True only for the selected model-day.")


## Output

The selected YAML is written to `configs/lr_search/` and copied to persistent
storage. A boundary winner is reported as a finite-range warning, not as a
global optimum.
